# Checking if tensor loads

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import json
from pathlib import Path

# Folder where the tensor notebook saved the outputs
OUTPUT_DIR = Path("/content/drive/MyDrive/solar_flare_forecasting/processed_tensors")

# Partition 1 files
npz_path = OUTPUT_DIR / "partition1_combined_clean.npz"
metadata_path = OUTPUT_DIR / "partition1_metadata_clean.csv"
features_path = OUTPUT_DIR / "partition1_feature_columns.json"

print("NPZ exists:", npz_path.exists(), npz_path)
print("Metadata exists:", metadata_path.exists(), metadata_path)
print("Feature columns exists:", features_path.exists(), features_path)

NPZ exists: True /content/drive/MyDrive/solar_flare_forecasting/processed_tensors/partition1_combined_clean.npz
Metadata exists: True /content/drive/MyDrive/solar_flare_forecasting/processed_tensors/partition1_metadata_clean.csv
Feature columns exists: True /content/drive/MyDrive/solar_flare_forecasting/processed_tensors/partition1_feature_columns.json


In [3]:
# Load Partition 1 tensor file
data = np.load(npz_path, allow_pickle=True)

# Show what arrays are inside the .npz file
print("Keys inside NPZ:")
print(data.files)

Keys inside NPZ:
['X', 'partition', 'y_flare', 'fl_nf_label', 'flare_class', 'harpnum', 'source_file', 'clean_keep', 'n_interpolated_rows', 'has_boundary_interpolation', 'xrquality_degraded']


In [4]:
# Load the main arrays
X = data["X"]
y_flare = data["y_flare"]

print("X shape:", X.shape)
print("y_flare shape:", y_flare.shape)
print("X dtype:", X.dtype)
print("y_flare dtype:", y_flare.dtype)

X shape: (73268, 60, 47)
y_flare shape: (73268,)
X dtype: float32
y_flare dtype: int8


In [5]:
# Check class counts
unique, counts = np.unique(y_flare, return_counts=True)

label_counts = pd.DataFrame({
    "label": unique,
    "count": counts
})

label_counts["class_name"] = label_counts["label"].map({
    0: "NF / no flare",
    1: "FL / flare"
})

label_counts

,label,count,class_name
0,0,72014,NF / no flare
1,1,1254,FL / flare


In [6]:
# Load metadata and feature columns
metadata = pd.read_csv(metadata_path)

with open(features_path, "r") as f:
    feature_columns = json.load(f)

print("Metadata shape:", metadata.shape)
print("Number of feature columns:", len(feature_columns))
print("First 10 features:", feature_columns[:10])
print("Last 5 features:", feature_columns[-5:])

Metadata shape: (73268, 26)
Number of feature columns: 47
First 10 features: ['TOTUSJH', 'TOTBSQ', 'TOTPOT', 'TOTUSJZ', 'ABSNJZH', 'SAVNCPP', 'USFLUX', 'TOTFZ', 'MEANPOT', 'EPSZ']
Last 5 features: ['XFLARE_LOC', 'XR_MAX', 'XR_QUAL', 'IS_TMFI', 'was_interpolated']


## Checking size of partition 1 tensor

### .npz file size on Google Drive

In [7]:
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/solar_flare_forecasting/processed_tensors")
npz_path = OUTPUT_DIR / "partition1_combined_clean.npz"

size_bytes = npz_path.stat().st_size
size_gb = size_bytes / (1024**3)

print(f"File size: {size_bytes:,} bytes")
print(f"File size: {size_gb:.3f} GB")

File size: 74,862,552 bytes
File size: 0.070 GB


### in-memory size of X

In [8]:
import numpy as np

data = np.load(npz_path, allow_pickle=True)
X = data["X"]

print("X shape:", X.shape)
print("X dtype:", X.dtype)
print(f"X memory size: {X.nbytes / (1024**3):.3f} GB")

X shape: (73268, 60, 47)
X dtype: float32
X memory size: 0.770 GB


## Checking colab ram

In [10]:
import psutil

ram_info = psutil.virtual_memory()
total_ram_gb = ram_info.total / (1024**3)
available_ram_gb = ram_info.available / (1024**3)

print(f"Total RAM: {total_ram_gb:.2f} GB")
print(f"Available RAM: {available_ram_gb:.2f} GB")

Total RAM: 12.67 GB
Available RAM: 10.60 GB
